# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and processing the [FAIR^2 Croissant](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
This dataset is described using a Croissant schema, accessible at the following URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and discover records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant metadata and dataset
dataset = mlc.Dataset(croissant_url)

# Show dataset metadata: title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Croissant ID: {dataset.metadata.identifier}")

## 2. Data Overview

List available record sets, along with their `@id`, title (`name`), and the field structure for each record set. All entities are referenced by their `@id` per best practices.

In [ ]:
# List all record sets in the dataset with @id, name, and fields
record_sets = dataset.metadata.record_sets
if not record_sets:
    raise ValueError("No record sets found in this dataset.")

for rs in record_sets:
    print(f"RecordSet: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print(f"  Fields (by @id):")
    for field in rs.fields:
        print(f"    - {field.id} (name: {field.name}, type: {field.data_type})")
    print("")

## 3. Data Extraction

Load data from each record set as a pandas DataFrame for analysis. _All entity references are by `@id`_.

In [ ]:
# Map of DataFrames by RecordSet ID
dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

for rsid in record_set_ids:
    # Load records lazily for the record set by id
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"RecordSet {rsid} loaded with shape: {df.shape}")

# Example: Show the first record set's DataFrame columns and head
main_rs_id = record_set_ids[0]
print(f"\nColumns for RecordSet {main_rs_id}:")
print(dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field (e.g., molecular weight or protein abundance) and perform:
- Filtering records by a numeric threshold
- Normalization (z-score)
- Grouping by a protein or sample annotation

> **NOTE:** The field IDs are referenced from the earlier overview. Update the variable assignments below based on the field structure from your overview above.

In [ ]:
# Pick a numeric field for analysis. Customize these @id strings to the dataset:
# E.g., 'accession', 'mw_kda', 'normalized_abundance', etc. (replace with matching field @id)

# Inspect available column names (fields) for the main record set
print(dataframes[main_rs_id].columns.tolist())

# Suppose the molecular weight field @id is 'mw_kda'
numeric_field_id = 'mw_kda'  # <-- Replace if needed, e.g., 'cr:mw_kda' if that's the actual @id
# Suppose grouping by sample type, e.g., 'sample' or 'accession'
group_field_id = 'accession'  # <-- Replace if needed

if numeric_field_id not in dataframes[main_rs_id].columns:
    raise ValueError(f"Column {numeric_field_id} not found in DataFrame. Update the field ID as per the overview above.")

df_main = dataframes[main_rs_id]
# Ensure numeric (could be string, so convert if necessary)
df_main[numeric_field_id] = pd.to_numeric(df_main[numeric_field_id], errors='coerce')

threshold = 10
filtered_df = df_main[df_main[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Z-score normalization
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, norm_col]].head())

# Group by a category
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGroup mean of {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field and the grouped means across different categories. Update the field names for your dataset if necessary.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Distribution plot for the numeric field (filtered)
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id} (> {threshold})")
plt.xlabel(numeric_field_id)
plt.show()

# If applicable, plot grouped means
if group_field_id in filtered_df.columns:
    plt.figure(figsize=(10, 5))
    group_plot_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    sns.barplot(x=group_field_id, y=numeric_field_id, data=group_plot_df)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- You have loaded the FAIR^2 dataset using `mlcroissant` and examined its structure using only Croissant `@id` references.
- The Data Overview illustrated how to discover available record sets and their fields in a programmatic way.
- Data extraction and EDA demonstrated how to select, filter, normalize, and group data for downstream analysis.
- Basic visualizations were produced to better understand distribution and grouping in the mass spectrometry data.

> Adapt variable names and `@id` selections per your dataset's overview above for more advanced queries and transformation.